### Lab 3.1: Basic Neural Network in PyTorch - Solution

Let's create a linear classifier one more time, but using PyTorch's automatic differentiation and optimization algorithms.  Then you will extend the perceptron into a multi-layer perceptron (MLP).

In [2]:
import numpy as np
import torch

We need to explicitly tell PyTorch when creating a tensor that we are interested in later computing its gradient

In [3]:
a = torch.tensor(5.,requires_grad=True)
a

tensor(5., requires_grad=True)

In [4]:
b = torch.tensor(6.,requires_grad=True)
c = 2*a+3*b
c

tensor(28., grad_fn=<AddBackward0>)

To extract the gradients, we first need to call `backward()`.

In [5]:
c.backward()

Now to get the gradient of any variable with respect to `c`, we simply access the `grad` attribute of that variable.

In [6]:
a.grad

tensor(2.)

In [7]:
b.grad

tensor(3.)

Let's load and format the Palmer penguins dataset for multi-class classification.

In [8]:
from palmerpenguins import load_penguins
from matplotlib import pyplot as plt

In [9]:
df = load_penguins()

# drop rows with missing values
df.dropna(inplace=True)

# get two features
X = df[['flipper_length_mm','bill_length_mm']].values

# convert species labels to integers
y = df['species'].map({'Adelie':0,'Chinstrap':1,'Gentoo':2}).values

To make the learning algorithm work more smoothly, we we will subtract the mean of each feature.

Here `np.mean` calculates a mean, and `axis=0` tells NumPy to calculate the mean over the rows (calculate the mean of each column).

In [10]:
X -= np.mean(X,axis=0)

Now we will convert our `X` and `y` arrays to torch Tensors.

In [11]:
X = torch.tensor(X).float()
y = torch.tensor(y).long()

In [12]:
from torch import nn

The `torch.nn.Sequential` class creates a feed-forward network from a list of `nn.Module` objects.  Here we provide a single `nn.Linear` class which performs an affine transformation ($Wx+b$) so that we will have a linear classifier.

In [13]:
linear_model = torch.nn.Sequential(
    torch.nn.Linear(2,3), # two inputs, three outputs
)

Now we create a cross-entropy loss function object and a stochastic gradient descent (SGD) optimizer.

In [14]:
loss_fn = torch.nn.CrossEntropyLoss()

In [15]:
lr = 1e-2
opt = torch.optim.SGD(linear_model.parameters(), lr=lr)

Finally we can iteratively optimize the model.

In [16]:
epochs = 100
for epoch in range(epochs):
    opt.zero_grad() # zero out the gradients

    z = linear_model(X) # compute z values
    loss = loss_fn(z,y) # compute loss

    loss.backward() # compute gradients

    opt.step() # apply gradients
    
    

    print(f'epoch {epoch}: loss is {loss.item()}')

epoch 0: loss is 1.8869390487670898
epoch 1: loss is 1.326594591140747
epoch 2: loss is 0.8547194004058838
epoch 3: loss is 0.5314492583274841
epoch 4: loss is 0.37310975790023804
epoch 5: loss is 0.31010422110557556
epoch 6: loss is 0.28248682618141174
epoch 7: loss is 0.267825186252594
epoch 8: loss is 0.2587156891822815
epoch 9: loss is 0.2523502707481384
epoch 10: loss is 0.24749529361724854
epoch 11: loss is 0.24354402720928192
epoch 12: loss is 0.24017184972763062
epoch 13: loss is 0.23719339072704315
epoch 14: loss is 0.23449698090553284
epoch 15: loss is 0.2320125252008438
epoch 16: loss is 0.22969412803649902
epoch 17: loss is 0.2275107204914093
epoch 18: loss is 0.2254406064748764
epoch 19: loss is 0.22346808016300201
epoch 20: loss is 0.22158139944076538
epoch 21: loss is 0.21977166831493378
epoch 22: loss is 0.21803177893161774
epoch 23: loss is 0.21635597944259644
epoch 24: loss is 0.21473954617977142
epoch 25: loss is 0.2131783813238144
epoch 26: loss is 0.211669057607650

### Exercises

Extend the above code to implement an MLP with a single hidden layer of size 100.


In [17]:
class MLP(nn.Module):
    def __init__(self, hidden_size=100):
        super(MLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(2, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 3)
        )
        
    def forward(self, x):
        return self.network(x)

In [18]:
mlp_model = MLP()
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.SGD(mlp_model.parameters(), lr=lr)

epochs = 100
for epoch in range(epochs):
    opt.zero_grad() # zero out the gradients

    z = mlp_model(X) # compute z values
    loss = loss_fn(z,y) # compute loss

    loss.backward() # compute gradients

    opt.step() # apply gradients

    print(f'epoch {epoch}: loss is {loss.item()}')

epoch 0: loss is 1.0047988891601562
epoch 1: loss is 0.48133841156959534
epoch 2: loss is 0.36889100074768066
epoch 3: loss is 0.30619505047798157
epoch 4: loss is 0.262914776802063
epoch 5: loss is 0.2332744151353836
epoch 6: loss is 0.21388980746269226
epoch 7: loss is 0.20161771774291992
epoch 8: loss is 0.19358104467391968
epoch 9: loss is 0.18774309754371643
epoch 10: loss is 0.18303902447223663
epoch 11: loss is 0.17901863157749176
epoch 12: loss is 0.17549018561840057
epoch 13: loss is 0.17235277593135834
epoch 14: loss is 0.16953930258750916
epoch 15: loss is 0.1669996976852417
epoch 16: loss is 0.16469430923461914
epoch 17: loss is 0.16259093582630157
epoch 18: loss is 0.16066350042819977
epoch 19: loss is 0.158890038728714
epoch 20: loss is 0.15725186467170715
epoch 21: loss is 0.15573345124721527
epoch 22: loss is 0.15432177484035492
epoch 23: loss is 0.15300554037094116
epoch 24: loss is 0.1517750322818756
epoch 25: loss is 0.15062186121940613
epoch 26: loss is 0.1495386511

Write code to compute the accuracy of each model.

Can you get the MLP to outperform the linear model?

In [19]:
def get_accuracy(model, X, y):
    model.eval()

    total = 0
    correct = 0

    with torch.no_grad():
        z = model(X)
        predictions = z.argmax(dim=1)
        correct += (y == predictions).sum()
        total += y.size(0)

    return float(correct / total)

print(f"Linear model accuracy: {100 * get_accuracy(linear_model, X, y):.2f}%")
print(f"MLP model accuracy: {100 * get_accuracy(mlp_model, X, y):.2f}%")

Linear model accuracy: 94.59%
MLP model accuracy: 95.20%


The MLP model was able to outperform the linear model in accuracy by 1.21